mempool.py
-----------
Transaction lifecycle queue. Financial/regulatory PoA validators require
FULL (unanimous) approval — there is no partial "majority passes" outcome.
A transaction that does not receive unanimous approval is NOT automatically
rejected; it is moved to UNDER_INSPECTION for manual/compliance review,
which is a distinct state from a final REJECTED (e.g. every validator
independently rejects the same hard rule violation).

States:
    PENDING           - submitted, awaiting validator votes
    UNDER_INSPECTION  - voted on, did not reach unanimous approval,
                        held for manual/compliance resolution
    APPROVED          - unanimous approval reached (directly, or via
                        inspection resolved as approved) — about to
                        execute on Ethereum
    CONFIRMED         - Ethereum confirmed the execution
    REJECTED          - final rejection (unanimous reject at vote time,
                        or an inspection resolved as rejected)

# 1. Mempool States

In [ ]:
import time

PENDING = "PENDING"
UNDER_INSPECTION = "UNDER_INSPECTION"
APPROVED = "APPROVED"
CONFIRMED = "CONFIRMED"
REJECTED = "REJECTED"

# 2. Mempool Class function

In [ ]:
class Mempool:
    def __init__(self):
        self._records = {}  # tx_id -> record

    def submit(self, tx_id: str, op_type: str, params: dict) -> dict:
        record = {
            "tx_id": tx_id, "op_type": op_type, "params": params,
            "status": PENDING, "decisions": [], "history": [],
            "submitted_at": time.time(),
        }
        self._records[tx_id] = record
        self._log(tx_id, PENDING, "submitted")
        return record

    def _log(self, tx_id, status, note):
        self._records[tx_id]["history"].append(
            {"status": status, "note": note, "timestamp": time.time()}
        )

    def record_votes(self, tx_id: str, decisions: list):
        self._records[tx_id]["decisions"] = decisions

    def mark_approved(self, tx_id: str, note: str = "unanimous approval reached"):
        self._records[tx_id]["status"] = APPROVED
        self._log(tx_id, APPROVED, note)

    def mark_under_inspection(self, tx_id: str, reason: str):
        self._records[tx_id]["status"] = UNDER_INSPECTION
        self._log(tx_id, UNDER_INSPECTION, reason)

    def mark_rejected(self, tx_id: str, reason: str):
        self._records[tx_id]["status"] = REJECTED
        self._log(tx_id, REJECTED, reason)

    def mark_confirmed(self, tx_id: str, eth_tx_hash: str):
        self._records[tx_id]["status"] = CONFIRMED
        self._records[tx_id]["eth_tx_hash"] = eth_tx_hash
        self._log(tx_id, CONFIRMED, f"Ethereum tx {eth_tx_hash}")

    def get(self, tx_id: str) -> dict:
        return self._records[tx_id]

    def list_by_status(self, status: str) -> list:
        return [r for r in self._records.values() if r["status"] == status]

    def pending(self): return self.list_by_status(PENDING)
    def under_inspection(self): return self.list_by_status(UNDER_INSPECTION)
    def rejected(self): return self.list_by_status(REJECTED)
    def confirmed(self): return self.list_by_status(CONFIRMED)

    def print_summary(self):
        print(f"Mempool — pending: {len(self.pending())}, "
              f"under_inspection: {len(self.under_inspection())}, "
              f"rejected: {len(self.rejected())}, confirmed: {len(self.confirmed())}")
